# Sprint 2 — Hindi + multilingual generalization

Confirm the XLSR-53 base (Hindi-capable) holds up on Hindi clips. Save to `results/sprint2_hindi_test.json`.

In [ ]:
# @title 1. Mount Drive + get the src/ package (results/checkpoints/test_data live on DRIVE)
from google.colab import drive
from pathlib import Path
import sys, os

drive.mount("/content/drive")

REPO_URL = "https://github.com/io-PEAK/VoxDetect.git"   # change if you forked
REPO_DIR = Path("/content/VoxDetect")                    # session-only git clone of the repo

# ---- Durable on Google Drive (persist across sessions) ----
# mirror folium: RESULTS_DIR, CHECKPOINT_DIR, data all under /content/drive/MyDrive/VoxDetect
ML_BASE  = Path("/content/drive/MyDrive/VoxDetect/ml-core")
RESULTS_DIR    = ML_BASE / "results"        # every evaluate run writes a UNIQUE json here
CHECKPOINT_DIR = ML_BASE / "checkpoints"     # fine-tuned/frozen model artifacts
TEST_DATA      = ML_BASE / "test_data"       # real/ + cloned/ clips, English + Hindi
results_dir = RESULTS_DIR; test_data_dir = TEST_DATA
for d in (RESULTS_DIR, CHECKPOINT_DIR, TEST_DATA):
    d.mkdir(parents=True, exist_ok=True)

# ---- git clone the repo (session-only) so we get the latest src/ ----
if not (REPO_DIR / "ml-core" / "src").exists():
    %cd /content
    !git clone --depth 1 {REPO_URL}
else:
    !git -C {REPO_DIR} pull --ff-only -q

SRC_PKG = REPO_DIR / "ml-core" / "src"
sys.path.insert(0, str(SRC_PKG))
print("src/ package at:", SRC_PKG)
print("RESULTS_DIR (Drive):", RESULTS_DIR)
print("CHECKPOINT_DIR (Drive):", CHECKPOINT_DIR)
print("TEST_DATA (Drive):", TEST_DATA)

# Install audio + ML deps (no stray 'audio' package)
!pip install -q torch torchaudio librosa soundfile transformers resemblyzer huggingface_hub numpy scipy
print("deps installed")

In [ ]:
# @title Run Hindi / multilingual generalization
import subprocess, sys
out = str(results_dir / "sprint2_hindi_test.json")
result = subprocess.run([
    sys.executable, "-m", "evaluate",
    "--root", str(test_data_dir),
    "--variant", "wav2vec2",
    "--out", out,
    "--find-threshold",
], cwd=str(SRC_PKG))
assert result.returncode == 0, "evaluate failed"
print("\nSaved ->", out, "(on Drive)")